In [ ]:
import os
from google.colab import drive

# CẤU HÌNH THÔNG TIN
# Public clone URL; Git handles any configured credentials.
REPO_OWNER = "theflyingkhui04"
REPO_NAME = "do-an-ptit-d22"
TARGET_DIR = f"/root/{REPO_NAME}"
CLONE_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

VSCODE_DRIVE_BACKUP = "/content/drive/MyDrive/vllm_colab_backup"


# BƯỚC 1: KẾT NỐI GOOGLE DRIVE
print("\n" + "="*60)
print("==> BƯỚC 1/4: KẾT NỐI GOOGLE DRIVE")
print("="*60)
drive.mount('/content/drive', force_remount=False)
os.makedirs(f"{VSCODE_DRIVE_BACKUP}/.vscode-cli", exist_ok=True)


# BƯỚC 2: ĐỒNG BỘ MÃ NGUỒN REPO
print("\n" + "="*60)
print("==> BƯỚC 2/4: ĐỒNG BỘ MÃ NGUỒN REPO TỪ GITHUB")
print("="*60)
if not os.path.exists(TARGET_DIR):
    !git clone {CLONE_URL} {TARGET_DIR}
else:
    !git -C {TARGET_DIR} pull

!chmod +x {TARGET_DIR}/scripts/*.sh


# BƯỚC 3: CÀI ĐẶT MÔI TRƯỜNG vLLM
print("\n" + "="*60)
print("==> BƯỚC 3/4: CÀI ĐẶT vLLM TRÊN SSD COLAB")
print("="*60)
!pip install -r {TARGET_DIR}/requirements-server.txt

print("\n--> Đang gỡ bỏ torchaudio để tránh xung đột CUDA...")
!pip uninstall -y torchaudio



# BƯỚC 4: THIẾT LẬP VS CODE CLI & KHỞI CHẠY TUNNEL
print("\n" + "="*60)
print("==> BƯỚC 4/4: KHỞI TẠO VS CODE TUNNEL 'colab-t4'")
print("="*60)

# 1. Tải CLI nếu chưa có (Sử dụng Direct Link của Microsoft để tránh lỗi DNS/Redirect trên T4 GPU)
if os.path.exists("./code"):
    print("--> Found existing VS Code CLI, removing for fresh install...")
    !rm -rf ./code
if os.path.exists("vscode_cli.tar.gz"):
    print("--> Found existing vscode_cli.tar.gz, removing for fresh download...")
    !rm -f vscode_cli.tar.gz

print("--> Downloading and extracting VS Code CLI (Direct Link)...")
!wget -O vscode_cli.tar.gz "https://update.code.visualstudio.com/latest/cli-alpine-x64/stable" && tar -xf vscode_cli.tar.gz

# 2. Liên kết thư mục lưu trữ thông tin đăng nhập với Drive
os.makedirs(f"{VSCODE_DRIVE_BACKUP}/.vscode-cli", exist_ok=True)
!ln -sfn {VSCODE_DRIVE_BACKUP}/.vscode-cli /root/.vscode-cli

# 3. Đăng nhập với cờ cố định là GitHub
print("--> Kiểm tra đăng nhập GitHub:")
!./code tunnel user login --provider github

# 4. Bật Tunnel
print("\n" + "="*60)
print("  ĐANG KHỞI CHẠY TUNNEL 'colab-t4'...")
print("="*60 + "\n")
!./code tunnel --accept-server-license-terms --name colab-t4